In [25]:
import cv2 as cv
import caer


import tensorflow as tf
from keras import layers
from keras import models
from keras import optimizers
from keras import saving
from keras.utils import image_dataset_from_directory #это генератор исходных данных, обязательно ознакомиться с документацией!
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
#from tensorflow import data as tf_data
from keras.utils import plot_model
import keras
from IPython.display import Image, display
from tensorflow.keras import regularizers

from tensorflow.keras.models import load_model


import pandas as pd
import os
import urllib.request

import random
import shutil

from huggingface_hub import hf_hub_download
from ultralytics import YOLO

from classification_models.tfkeras import Classifiers

In [ ]:
ResNet18, preprocess_input = Classifiers.get('resnet18')
model_resnet1 = ResNet18(input_shape=(224, 224, 3), weights='imagenet', include_top=False)


model_name = 'model_resnet_new_one_again1.keras'
url = 'https://www.dropbox.com/scl/fi/quok4ymxq21ip7zdce0h6/model_resnet_new_one_again1.keras?rlkey=glvzbjc1oyktkkaw95nzkdg3u&st=o2i2xr45&dl=1'
if not os.path.exists(model_name):
    urllib.request.urlretrieve(url,model_name)
model = tf.keras.models.load_model('model_resnet_new_one_again1.keras')

In [ ]:
model_path_yolo = hf_hub_download(repo_id="AdamCodd/YOLOv11n-face-detection", filename="model.pt")
model_yolo = YOLO(model_path_yolo)

In [ ]:
def label_translate(label):
    class_names=['angry','disgust','fear','happy','neutral','sad','surprise']
    idx=np.argmax(label)
    return class_names[idx]

In [ ]:
EMOTION_LABELS = ['angry','disgust','fear','happy','neutral','sad','surprise']

analytics_data = []

class CentroidTracker:
    def __init__(self, max_distance=80):
        self.next_id = 1
        self.current_objects = dict()
        self.max_distance = max_distance

    def update(self, faces_rect):
        new_objects = dict()
        for box in faces_rect:
            x, y, w, h = box
            cx, cy = int(x + w / 2), int(y + h / 2)
            assigned_id = None
            min_dist = self.max_distance
            
            for obj_id, (prev_cx, prev_cy) in self.current_objects.items():
                dist = np.hypot(cx - prev_cx, cy - prev_cy)
                if dist < min_dist:
                    assigned_id = obj_id
                    min_dist = dist
            
            if assigned_id is None:
                assigned_id = f"person_{self.next_id}"
                self.next_id += 1
                
            new_objects[assigned_id] = (cx, cy, w, h)
            
        self.current_objects = {obj_id: (b[0], b[1]) for obj_id, b in new_objects.items()}
        return new_objects

def save_frame_analytics(frame_id, obj_id, raw_scores):
    log_row = {'Frame': frame_id, 'ID': obj_id}
    for i, label in enumerate(EMOTION_LABELS):
        log_row[label] = float(raw_scores[i])
    analytics_data.append(log_row)

def plot_local_results():
    if not analytics_data:
        print("Нет данных для графиков.")
        return
        
    df = pd.DataFrame(analytics_data)
    df.to_csv('video_analytics_report.csv', index=False)
    print("результаты сохранены в 'video_analytics_report.csv'")

    for obj_id in df['ID'].unique():
        person_df = df[df['ID'] == obj_id].sort_values(by='Frame')
        
        plt.figure(figsize=(10, 4))
        for label in EMOTION_LABELS:
            if label in person_df.columns:
                plt.plot(person_df['Frame'], person_df[label], label=label)
        
        plt.title(f"анализ изменения эмоций для: {obj_id}")
        plt.xlabel("кадры (время)")
        plt.ylabel("уверенность модели (0.0 - 1.0)")
        plt.legend()
        plt.grid(True)
        plt.show()

In [18]:
print('из файла/в реальном времени (нужное Ctrl+C -> Ctrl+V)')
choice=input()

из файла/в реальном времени (нужное Ctrl+C -> Ctrl+V)


In [ ]:
if choice=='из файла':
    video_path = input('Полный путь к файлу: ')
    video_path.replace('\\', '/') 
    capture = cv.VideoCapture(video_path)
else:
    capture = cv.VideoCapture(0)

tracker=CentroidTracker(max_distance=100)
frame_id=0

if not capture.isOpened():
    print('не смог открыть видеофайл')  
while True:
    is_true, frame = capture.read()

    if not is_true or frame is None:
        print("конец видео или проблема с кадром")
        break
    frame_id+=1
    try:
        pred = model_yolo(frame, stream=True, conf=0.25)
        
        faces_rect_for_tracker = []
        yolo_boxes = []

        for face in pred:
            for box in face.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                w = x2 - x1
                h = y2 - y1
                faces_rect_for_tracker.append([x1, y1, w, h])
                yolo_boxes.append((x1, y1, x2, y2))
        tracked = tracker.update(faces_rect_for_tracker)

        for (x1, y1, x2, y2) in yolo_boxes:
            cx = int(x1 + (x2 - x1) / 2)
            cy = int(y1 + (y2 - y1) / 2)
            
            current_id = "Unknown"
            
            for obj_id, (tcx, tcy, tw, th) in tracked.items():
                if abs(cx - tcx) < 40 and abs(cy - tcy) < 40: 
                    current_id = obj_id
                    break

            pic = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            h_f, w_f, _= frame.shape
            crop = frame[max(0, y1):min(h_f, y2), max(0, x1):min(w_f, x2)]
            
            if crop.size == 0:
                continue
                
            pic = cv.resize(crop, (224, 224))
            pic = np.expand_dims(pic, axis=0).astype('float32')
            pic = preprocess_input(pic) 
            label = model.predict(pic)

            save_frame_analytics(frame_id, current_id, label[0])

            display_text = f"{current_id}: {label_translate(label)}"
            cv.putText(frame, text=display_text, org=(x1, y1 - 10), fontFace=cv.FONT_HERSHEY_COMPLEX, fontScale=0.5, color=(0, 255, 0), thickness=2)
            cv.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv.imshow('video', frame)
        
    except Exception as e:
        print(f"косяк с кадрами: {e}")
        break
        
    if cv.waitKey(20) & 0xFF == ord('d'):
        break

capture.release()
cv.destroyAllWindows()
plot_local_results()


0: 320x640 14 faces, 50.4ms
Speed: 2.2ms preprocess, 50.4ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step

0: 320x640 14 faces, 196.8ms
Speed: 7.8ms preprocess, 196.8ms inference, 2.8ms postprocess per image at shape (1, 3, 320, 640)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━

KeyboardInterrupt: 